# Chapter 3: Environmental Standardization of Zoobenthic Community Composition Using Reference-Site Taxa Clusters

**Objectives**

This chapter sets out to **control environmental confounding** in zoobenthic community composition. It does this by identifying the natural community types that occur at clean reference sites, then linking those community types to the environmental conditions they belong to — so that for any site, we can say which community would naturally be expected there, independent of contamination effects.

**Reasons**

A community at any given site is shaped by two main forces: the local environment (geology, hydrology, climate, habitat) and contamination. To study the effects of contamination cleanly, we need a way to set the environmental influence aside. The straightforward approach — putting both environment and contamination into the same statistical model — runs into trouble: the variables overlap, the model becomes hard to interpret, and the ecological meaning gets diluted.

A more ecological approach is to group sites by the community they would have **without contamination** — that is, by their natural community type. Once these natural types are established, environmental conditions can be linked to them through a simple mapping rather than a tangled regression, and contamination can be studied as a deviation from the natural expectation.

**How**

Three steps deliver this approach. First, the reference sites identified in Chapter 2 — the cleanest sites — are grouped by similarity of their taxa composition, producing a small number of distinct **reference community types**. Second, only the reference sites whose taxa-based grouping is also supported by their environmental conditions are kept as training cases; this filtering step removes ambiguous sites where the biology and the environment disagree on which community type fits best. Third, a classifier is trained on these high-confidence sites to learn the mapping from environmental conditions to community types, and is then applied to **all 310 sites** — including contaminated ones — to assign each its expected community type based on environment alone.

This indirectly partitions the environmental descriptor space into a handful of community-type regions. Although it **discretizes** what is in reality a continuous environmental gradient, the trade-off buys **clarity**: each site can be compared to others that share an expected community type, even if their actual measured communities differ because of contamination.

**Expected Contributions**

- A small set of **reference community types**, each statistically distinct and robust, identified from the cleanest sites in the dataset.
- A trained **environmental classifier** that links environmental conditions to expected community types, with its decision boundaries anchored on sites where biology and environment agree.
- Every one of the 310 sites — clean or contaminated — receives an **expected community type** (or several, if its environment is ambivalent), providing the comparison baseline that downstream chapters use to quantify contamination effects on community composition.

**List of Frameworks**

This chapter contains three frameworks that build the standardization in sequence:

- **Framework 1 — Reference community types from least-stressed sites.** Groups the reference sites by taxa composition into a small number of natural community types and confirms the groups are statistically distinct and stable.
- **Framework 2 — Finding cross-view-supported reference sites.** Filters the reference sites down to those whose community-type label is supported by both taxa composition *and* environmental conditions, producing the high-confidence training set for the classifier.
- **Framework 3 — Training, diagnosing, and applying the environmental classifier.** Builds the classifier on the high-confidence training set, evaluates it against held-out cases, and applies it to all 310 sites to assign each its expected community type from environmental conditions.

## Framework 1: Reference Community Types from Least-Stressed Sites

To define naturally occurring community types without imposing descriptors on the clustering process, unconstrained taxa-based clustering is applied to the least-stressed sites.

**Questions and Motivations**

Community composition is influenced by at least both natural environmental variation and contamination stressors, and other unmeasured factors, etc.

In studying the quantitative effects of contamination on community composition, a direct regression of community composition on both environmental covariates and chemical stressors may improve statistical fit, but it risks collinearity, reduced interpretability, and weaker ecological meaning.

Instead of treating community composition as a continuous response to many covariates, this framework asks whether the least-stressed sites contain distinct reference community types. This reflects a more **ecological view**: even without contamination, zoobenthic assemblages may occur as several recognizable community types, each with its own characteristic taxa composition.

> In this study, it simply frames the all possible factors influencing community composition into **two** categories: natural factors and contamination stressors. This simplification gives the rationale that 'least-polluted sites hold the approximation to natural community composition'.

Sites belonging to the same community type are expected to have similar taxa composition. The full ecological profile defining a community type is never completely observable. However, some measured descriptors, such as depth, temperature, dissolved oxygen, or substrate features, may partially **describe** why certain taxa assemblages occur together.

> The core philosophy is that a community type is **defined** first by taxa composition, not by the limited environmental descriptors we happen to measure. Environmental descriptors may help **describe** the community types after they are identified, but they should not define the types at the clustering stage.

Therefore, it is important to avoid defining community types solely from partial descriptors. If community groups are constrained by these descriptors from the beginning, the resulting typology may reflect the structure of the measured covariates **rather than** the structure of the biological assemblages themselves. Such a constrained typology may be statistically convenient, but it can mask meaningful taxa-based community patterns and weaken the ecological motivation of identifying true taxa assemblages.

For this reason, the reference community typology is estimated using **unconstrained** clustering on the taxa matrix only. The analysis focuses on the least-stressed sites because they provide the best available approximation to a natural or minimally disturbed state. The taxa compositions observed at these sites are treated as empirical instances from which the underlying reference community types can be estimated.

**Methodology**

Select the $m$ least-stressed sites as reference sites based on a contamination score $\mathbf{c}$ defined in Framework 2 of Chapter 2. These reference sites serve as the best available natural approximation to community composition under the preferred toxicological view.

Apply unconstrained hierarchical clustering to the $m$ reference sites using their taxa matrix $\mathbf{T_{ref}}$. The clustering yields $k$ clusters along with $k-1$ linkage values, each representing a cluster separation metric at a bifurcation that produces these $k$ clusters (with $k_{\text{max}} = m$). Each of the finalized $k$ clusters represents a recognizable reference community type.

Because sequential merging is highly sensitive to the initial site configuration, statistical evidence is needed to determine whether the chosen $k$ clusters are **distinct** from one another and **robust** within themselves. When both distinctiveness and robustness are statistically supported, the retained $k$ clusters are defensible.

**Conceptual Steps**

1. Transform the reference taxa matrix into $\mathbf{T_{ref}'}$ and validate that the transformation preserves ecological meaning throughout the clustering process. Specifically, the following metrics — defined at progressively higher measurement levels — must remain ecologically meaningful:
   - distance between sites within a single taxon variable
   - dissimilarity between sites aggregated over the 16 taxon variables
   - linkage between clusters

2. Construct the Manhattan (city-block) dissimilarity matrix $\mathbf{D_{ref}}$ from the transformed reference taxa matrix $\mathbf{T_{ref}'}$. City-block distance reflects differences in overall taxa composition between sites more faithfully than Euclidean distance. The pairwise dissimilarity between sites $i$ and $j$ is computed as:

$$
\mathbf{D}_{ij} = \sum_{k=1}^{16} |T'_{ik} - T'_{jk}|
$$

Applying this to all pairs of sites in $\mathbf{T_{ref}'}$ yields the city-block pairwise dissimilarity matrix:

$$
\mathbf{D_{ref}} \in \mathbb{R}^{m \times m}
$$

3. Apply Ward's hierarchical clustering to the city-block dissimilarity matrix $\mathbf{D_{ref}}$ to generate a dendrogram of the $m$ reference sites. Reading the dendrogram from **top to bottom**, the tree grows through a series of bifurcations; each bifurcation induces a linkage value that quantifies the separation between the two resulting branches, and the leaves descending from each branch form a cluster. This process repeats $k-1$ times until every site forms its own cluster, where $k_{\text{max}} = m$.

   Formally, let $\mathcal{C}^{(t)} = \{C_1^{(t)}, C_2^{(t)}, \dots, C_{m-t}^{(t)}\}$ denote the partition of sites at **agglomeration** step $t$, where $t = 0, 1, \dots, m-1$. The initial partition is $\mathcal{C}^{(0)} = \{\{1\}, \{2\}, \dots, \{m\}\}$. At each step, the two clusters minimizing Ward's linkage criterion are merged:

$$
(C_p^*, C_q^*) = \arg\min_{C_p, C_q \in \mathcal{C}^{(t)}} \, \Delta(C_p, C_q)
$$

   where the Ward linkage increment is

$$
\Delta(C_p, C_q) = \frac{|C_p|\,|C_q|}{|C_p| + |C_q|} \, d(\bar{C}_p, \bar{C}_q)
$$

   with $d(\cdot, \cdot)$ the city-block distance applied to cluster centroids $\bar{C}_p$ and $\bar{C}_q$, and $|\cdot|$ denoting cluster size. The sequence of linkage values $\{\Delta_1, \Delta_2, \dots, \Delta_{m-1}\}$ defines the dendrogram heights.

> Ward's clustering is an **agglomerative** hierarchical clustering that begins with individual sites as singleton clusters and recursively merges the two closest clusters at each step. The bifurcation framing is used to visually describe and interpret the clustering structure — it is not a computational step in the algorithm.

4. Apply node-wise ANOVA to **each bifurcation node** to test whether the two resulting branches differ significantly in taxa composition.

   Formally, at a given bifurcation node splitting cluster $C$ into subclusters $C_A$ and $C_B$ (with $|C_A| = n_A$, $|C_B| = n_B$, and $n = n_A + n_B$), define for each taxon variable $k \in \{1, 2, \dots, 16\}$ the within-group and between-group sums of squares:

$$
SS_{\text{between}}^{(k)} = n_A (\bar{T}'_{A,k} - \bar{T}'_{C,k})^2 + n_B (\bar{T}'_{B,k} - \bar{T}'_{C,k})^2
$$

$$
SS_{\text{within}}^{(k)} = \sum_{i \in C_A} (T'_{ik} - \bar{T}'_{A,k})^2 + \sum_{i \in C_B} (T'_{ik} - \bar{T}'_{B,k})^2
$$

   The F-statistic at the node is

$$
F^{(k)} = \frac{SS_{\text{between}}^{(k)} / (2 - 1)}{SS_{\text{within}}^{(k)} / (n - 2)}
$$

   with associated p-value $p^{(k)}$. The F-statistic and p-value serve as measures of node-wise distinctiveness between branches.

5. Apply `pvclust` to the **finalized** $k$ clusters to evaluate the within-cluster robustness of each cluster through multiscale bootstrap resampling.

   For each cluster $C_j$ ($j = 1, 2, \dots, k$), `pvclust` performs $B$ bootstrap replicates at multiple sample-size scales $r \in \{r_1, r_2, \dots, r_R\}$, where each replicate **resamples the taxa variables** and reruns the hierarchical clustering. Let $BP_j(r)$ denote the bootstrap probability that cluster $C_j$ appears in the resampled dendrogram at scale $r$. The approximately unbiased (AU) p-value is obtained by fitting a regression of $\Phi^{-1}(BP_j(r))$ on $\sqrt{r}$ and $1/\sqrt{r}$ and extrapolating:

$$
AU_j = 1 - \Phi\!\left(\hat{\beta}_0 - \hat{\beta}_1\right)
$$

   where $\Phi$ is the standard normal CDF, and $\hat{\beta}_0, \hat{\beta}_1$ are the fitted regression coefficients. A high $AU_j$ (typically $AU_j \geq 0.95$) indicates that cluster $C_j$ is robustly recovered across resampling and is thus statistically supported.

**Expected Conclusions**

- Ward's hierarchical clustering on the transformed taxa data of $m$ reference sites supports $k$ clusters with high linkage values, proper size of descending leaves.

- Node-wise ANOVA shows the bifurcations that produce the $k$ clusters are statistically significant over a large proportion of taxon variables.

- `pvclust` shows that the $k$ clusters are robustly recovered across multiscale bootstrap resampling, with high AU p-values.

### Computational Process

#### Inputs

**Metadata**

- taxa data $\mathbf{T}$

**Artifacts**

- table <span style="color: blue;">A2</span>: table of reference subset membership indicator
   
   - rows: 
   $(\text{site ID}, \text{if reference})$
   
   - shapes: $(310 \times 2)$

  - source: output artifact from the random-subset validation framework

#### Process

**Transformations**

- none; $\mathbf{T}$ is already in proportional octave format and requires no further transformation before dissimilarity computation.

Bridging tools:

- *boolean subset selector*

  - It takes a dataframe with at least a boolean column $\text{if reference}$, selects the rows flagged `True` in that column, and outputs the reference subset $\mathbf{T}_{\mathrm{ref}}$.

**Modeling**

- *city-block dissimilarity matrix constructor*

  - It accepts the reference taxa matrix $\mathbf{T}_{\mathrm{ref}}$ and computes the city-block pairwise dissimilarity matrix, outputting $\mathbf{D}_{\mathrm{ref}} \in \mathbb{R}^{m_{\mathrm{ref}} \times m_{\mathrm{ref}}}$.

- *Ward's hierarchical clustering*

  - It accepts the city-block dissimilarity matrix $\mathbf{D}_{\mathrm{ref}}$ and applies Ward's agglomerative hierarchical clustering. It outputs the full clustering hierarchy: the $k$ cluster partitions for $k = 1, 2, \dots, m_{\mathrm{ref}}$, the corresponding $m_{\mathrm{ref}} - 1$ linkage values $\{\Delta_1, \Delta_2, \dots, \Delta_{m_{\mathrm{ref}}-1}\}$, and the descending leaves of each cluster at every bifurcation.

- *node-wise ANOVA*

  - It accepts the two branches $C_A$ and $C_B$ at each bifurcation node of the dendrogram and fits a one-way ANOVA per taxon variable, testing whether the two branches differ in taxa composition.

- *finalized-cluster ANOVA*

  - It accepts the finalized $k$ clusters and fits a one-way ANOVA per taxon variable across all $k$ groups, testing whether the retained clusters differ in taxa composition.

- *pvclust bootstrap resampling*

  - It accepts the reference taxa matrix $\mathbf{T}_{\mathrm{ref}}$ and repeatedly resamples taxa variables at multiple sample-size scales, refitting Ward's clustering on city-block dissimilarities at each replicate to assess the within-cluster robustness of the finalized $k$ clusters.

**Interpreters**

- *cluster dendrogram renderer*

  - It accepts the Ward's clustering hierarchy and draws the dendrogram with bifurcations annotated by linkage values, used to visually identify the finalized $k$ clusters.

- *node-wise ANOVA summarizer*

  - It accepts each fitted node-wise ANOVA model and returns the per-taxon F-statistic and p-value, used as measures of branch-level distinctiveness at each bifurcation.

- *finalized-cluster ANOVA summarizer*

  - It accepts each fitted finalized-cluster ANOVA model and returns the per-taxon F-statistic and p-value, used as measures of overall distinctiveness across the retained $k$ clusters.

- *AU p-value extractor*

  - It accepts each fitted pvclust object and extracts the approximately unbiased (AU) p-value per cluster, used as the main measure of within-cluster robustness.

#### Outputs

**Results**

- table: node-wise ANOVA results for the first $k-1$ bifurcation nodes in the dendrogram
  - rows: $(\text{split ID}, n_{\text{left}}, n_{\text{right}}, \text{taxon}, \bar{x}_{\text{left}}, \bar{x}_{\text{right}}, F, p, p_{\text{FDR}})$
  - shape: $((k-1) \times n_{\text{sig taxa}}) \times 9$

- table: finalized-cluster ANOVA results and approximately unbiased (AU) p-values across the $k$ retained clusters
  - rows: $(\text{taxon}, SS_{\text{between}}(df), SS_{\text{within}}(df), F, p, \bar{x}_{C_1}, \ldots, \bar{x}_{C_k})$
  - shape: $(16 + 2) \times (5 + k)$

- figure: Ward's dendrogram of the $m_{\mathrm{ref}}$ reference sites
  - x-axis: site labels of the $m_{\mathrm{ref}}$ reference sites, leaves colored by the $k$ cluster labels
  - y-axis: linkage value $\Delta$ at each bifurcation
  - annotations: AU p-value annotated at each of the $k-1$ bifurcation nodes producing the finalized clusters
  - legend: colors for the $k$ clusters

- figure: barplots of the 16 taxa relative concentrations across the $k$ clusters
  - x-axis: 16 taxon ticks, each holding $k$ bars for the $k$ clusters
  - y-axis: mean relative concentration $\bar{p}_{jk}$ of taxon $j$ in cluster $k$
  - annotations: asterisks marking taxa with significant ANOVA p-values across the $k$ clusters
  - legend: colors for the $k$ clusters

**Artifacts**

- table <span style="color: blue;">A3</span>: table of only reference sites with their cluster labels
  - rows: $(\text{site ID}, \text{cluster label})$
  - shape: $m \times 2$

#### Demonstration Outputs

<div style="width: 50%; max-width: 700px; margin: 0 auto;">
  <img src="demos_images/ward_dendrogram.png" alt="Ward Dendrogram with AUs" style="width: 100%; display: block;">
  <p style="margin-top: 8px;"><em>Figure: Ward's dendrogram with approximately unbiased (AU) p-values annotated at the bifurcation nodes producing the 3 finalized clusters.</em></p>
</div>

## Framework 2: Finding Cross-View-Supported Reference Sites

**If sites are clustered by environmental descriptors, do they tend to group with sites from the same taxa-defined cluster?**

**Questions and Motivations**

The reference cluster labels $\{C_1, C_2, \dots, C_k\}$ obtained from Ward's clustering are defined entirely in **taxa space** — they reflect biological similarity among sites without any reference to environmental descriptors. When these labels are subsequently used as response classes for training an environmental classifier, an implicit assumption is invoked: **that sites grouped by taxa similarity are also coherent in environmental descriptor space**. This assumption does not always hold.

A taxa-defined cluster $C_i$ may appear compact in taxa space but become dispersed when projected into environmental space. Some sites assigned to $C_i$ may sit closer, environmentally, to members of a different cluster $C_j$. Such sites are **discordant** between the two views: their taxa membership and their environmental position disagree on which community type they belong to. Discordance can arise from at least three sources:

- the measured environmental descriptors are **insufficient** to distinguish the taxa-defined clusters;
- the taxa-defined cluster boundaries are themselves uncertain near the dendrogram's bifurcation thresholds;
- genuine ecological **mismatch** at the site, where local biotic factors decouple community composition from the measured abiotic environment.

If all sites in $C_i$ are treated as equally informative training cases, the classifier is forced to fit environmental features to taxa-based labels at sites where the two views **disagree**. From the optimization standpoint, these discordant sites behave *like* label noise — they push the decision boundary in directions **inconsistent** with the bulk of the cluster — even though the labels themselves are not erroneous in any absolute sense, since no ground-truth community type exists outside the clustering procedure. The effect is to inflate apparent classifier error in a way that conflates two distinct issues: insufficient environmental descriptors and ambiguous cluster membership.

To address this, this framework identifies **env–taxa concordant sites** — the subset of each $C_i$ whose membership is supported in *both* taxa space and environmental space — via **consensus clustering** across the two views, using the Ward's taxa-based labels as the **benchmark** partition. Only concordant sites are used as training cases for the environmental classifier, so the learned mapping rests on sites where the two data views agree on cluster membership.


<div style="width: 80%; max-width: 700px; margin: 0 auto; text-align: center;">

<img src="demos_images/idea_concordant_sites2.png" alt="Conceptual illustration of taxa-environment concordance" style="width: 100%; display: block; margin: 0 auto;">

</div>

<div style="width: 80%; max-width: 700px; margin: 8px auto 0 auto; text-align: left; font-size: 14px; line-height: 1.5;">

<em><strong>Conceptual illustration of taxa–environment concordance for classifier training.</strong></em>

<strong>Left:</strong> a taxa-defined cluster $C_i$ in taxa space may become more dispersed when projected into environmental descriptor space, indicating that not all taxa-cluster members are environmentally coherent. The black circle marks the intersection: the env–taxa concordant subset of $C_i$, whose members remain stably clustered with their anchor siblings in both views.

<strong>Right:</strong> applying this criterion across all $k=3$ anchor clusters identifies core sites whose taxa membership is supported by environmental similarity. These concordant sites provide training cases whose anchor labels are stable in both feature views, supporting the modeling of the mapping from environmental descriptors to reference community types.

</div>

**Methodology**

The finalized $k$ reference clusters $\{C_1, C_2, \dots, C_k\}$ from the Ward's hierarchical clustering framework serve as the **anchor partition**. They define, for each site $i \in C_j$, two reference sets: its **siblings** $\{i' : i' \in C_j, i' \neq i\}$ and its **non-siblings** $\{i' : i' \in C_l, l \neq j\}$. The anchor partition is treated as a benchmark, not as ground truth — its role is to provide a reference grouping against which each site's clustering stability can be probed in two independent feature views.

Two independent consensus clustering procedures are then run on the $m_{\mathrm{ref}}$ reference sites: one in **taxa space** using $\mathbf{T_{ref}'}$, and one in **environmental space** using $\mathbf{E_{ref}'}$. Each procedure repeatedly resamples the input matrix, refits the clustering algorithm, and accumulates a **co-assignment matrix** $\mathbf{M}^{T}, \mathbf{M}^{E} \in [0,1]^{m_{\mathrm{ref}} \times m_{\mathrm{ref}}}$, where each entry $M_{ii'}$ records the empirical frequency with which sites $i$ and $i'$ are placed in the same cluster across resampling iterations.

From each co-assignment matrix, a per-site **concordance margin** is computed against the anchor partition. For site $i \in C_j$, the margin in view $V \in \{T, E\}$ is:

$$
\mu_i^{(V)} = \underbrace{\frac{1}{|C_j| - 1} \sum_{i' \in C_j, i' \neq i} M_{ii'}^{(V)}}_{\text{mean co-assignment with siblings}} \; - \; \underbrace{\max_{l \neq j} \frac{1}{|C_l|} \sum_{i' \in C_l} M_{ii'}^{(V)}}_{\text{strongest mean co-assignment with non-siblings}}
$$

A non-negative margin in view $V$ means site $i$ co-clusters more strongly with its anchor siblings than with any rival cluster *in that view* — its anchor label is stable under resampling in $V$. A negative margin means a rival cluster pulls $i$ more strongly than its own anchor siblings — the anchor label is fragile in $V$.

Sites are then classified by the joint sign of their margins across the two views:

$$
\text{status}(i) = \begin{cases}
\textbf{concordant} & \mu_i^{(T)} \geq 0 \;\text{and}\; \mu_i^{(E)} \geq 0 \\
\textbf{taxa-only stable} & \mu_i^{(T)} \geq 0 \;\text{and}\; \mu_i^{(E)} < 0 \\
\textbf{env-only stable} & \mu_i^{(T)} < 0 \;\text{and}\; \mu_i^{(E)} \geq 0 \\
\textbf{ambiguous} & \mu_i^{(T)} < 0 \;\text{and}\; \mu_i^{(E)} < 0 \\
\end{cases}
$$

Concordant sites form the **high-confidence training set** for downstream environmental classifier modeling: their anchor cluster membership is supported under resampling in both taxa space *and* environmental space, so the supervised mapping $\mathbf{E} \rightarrow C_j$ is trained on sites where the two views agree. The remaining three categories are not discarded — they are retained as interpretable diagnostics:

- **taxa-only stable** sites suggest that environmental descriptors fail to recover the taxa structure at that site (a descriptor-insufficiency signal);
- **env-only stable** sites suggest that the site sits near a taxa-cluster boundary and would have been more naturally grouped with a different anchor cluster on environmental grounds (a taxa-label fragility signal);
- **ambiguous** sites are unstable in both views simultaneously, and likely represent transitional or atypical communities that are not well captured by either descriptor type.

This four-category labeling preserves all information while restricting classifier training to the concordant subset, so that the resulting environment-to-taxa mapping is anchored on cross-view-supported cases and the remaining sites contribute as a structured diagnostic **rather than** as unexplained noise.

**Conceptual Steps**

1. Take the $m_{\mathrm{ref}}$ reference sites and their finalized $k$ cluster labels from the output of Framework 1, and adopt the taxa cluster labels as the **anchor partition** for the consensus clustering procedure.

2. Specify the **underlying clustering procedure** in each view. The taxa view reuses the Ward's hierarchical clustering on city-block dissimilarities of proportional octave taxa features from Framework 1. The environmental view applies Ward's hierarchical clustering to Euclidean dissimilarities of z-score standardized environmental descriptors.

3. **Resample** the input matrix of view $V$ over $B$ iterations. At each iteration, refit the underlying clustering procedure on the resampled matrix. For each pair of sites $(i, q)$, record the proportion of iterations in which both sites are sampled *and* placed in the same cluster — this entry forms $M_{iq}^{(V)}$ of the co-assignment matrix $\mathbf{M}^{(V)} \in [0,1]^{m_{\mathrm{ref}} \times m_{\mathrm{ref}}}$.

4. For each site $i \in C_j$, average its co-assignment values with its anchor **siblings** in $C_j$, and average its co-assignment values with the **non-siblings** in each rival cluster $C_l, l \neq j$. The **concordance margin** $\mu_i^{(V)}$ is the difference between the sibling average and the strongest non-sibling average:

$$
\mu_i^{(V)} = \frac{1}{|C_j| - 1} \sum_{\substack{i' \in C_j \\ i' \neq i}} M_{ii'}^{(V)} \; - \; \max_{l \neq j} \, \frac{1}{|C_l|} \sum_{i' \in C_l} M_{ii'}^{(V)}
$$

5. Jointly evaluate the signs of $\mu_i^{(T)}$ and $\mu_i^{(E)}$ to partition the taxa–environment concordance space into four quadrants — **concordant**, **taxa-only stable**, **env-only stable**, and **ambiguous** — and assign each site to one quadrant based on the sign pair of its margins. This concordance space of four quadrants can be visualized as a scatterplot of $\mu_i^{(T)}$ vs. $\mu_i^{(E)}$.

**Expected Conclusions**

- The $m_{\mathrm{ref}}$ reference sites distribute across the four quadrants of the taxa–environment concordance space. A substantial proportion fall into the **concordant** quadrant, where anchor labels are jointly supported by both views — these sites form a high-confidence training set on which the supervised mapping from environmental descriptors to taxa cluster labels can be modeled with minimal cross-view contradiction.

- The remaining sites in the three discordant quadrants do not fail the framework — they act as structured diagnostics, exposing the limits of representing a continuous ecological gradient through a discrete taxa-cluster typology, and indicate where the current set of environmental descriptors and taxa-clustering choices may need to be revisited.

### Computation Process

#### Inputs

**Metadata**

- Taxa data $\mathbf{T}$
- Environmental data $\mathbf{E}$

**Artifacts**

- table <span style="color: blue;">A3</span>: table of only reference sites with their cluster labels
  - rows: $(\text{site ID}, \text{cluster label})$
  - shape: $m \times 2$

#### Process

**Transformations**

- *z-score standardizer*

  - It takes the environmental descriptor matrix $\mathbf{E}$ and applies z-score standardization, so descriptor scales are comparable under Euclidean dissimilarity.

**Bridging tools**

- *boolean subset selector*

  - It takes a dataframe with at least a boolean column $\text{if reference}$, selects the rows flagged `True` in that column, and outputs the reference subset $\mathbf{T}_{\mathrm{ref}}$ or $\mathbf{E}_{\mathrm{ref}}$.

**Modeling**

- *subset resampler*

  - It accepts a matrix $\mathbf{X} \in \mathbb{R}^{m_{\mathrm{ref}} \times n}$ and a number of iterations $B$, and resamples the rows of $\mathbf{X}$ with replacement at each iteration, yielding $B$ resampled matrices $\{\mathbf{X}^{(1)}, \mathbf{X}^{(2)}, \dots, \mathbf{X}^{(B)}\}$.

- *city-block dissimilarity matrix constructor*

  - It functions as in Framework 1. Here it is invoked on each resampled taxa matrix $\mathbf{T}_{\mathrm{ref}}^{(b)}$ and outputs the resampled dissimilarity matrix $\mathbf{D}_{\mathrm{ref}}^{(T;b)}$ for the next reclustering step.

- *euclidean dissimilarity matrix constructor*

  - It accepts each resampled environmental matrix $\mathbf{E}_{\mathrm{ref}}^{(b)}$ and computes the Euclidean pairwise dissimilarity matrix $\mathbf{D}_{\mathrm{ref}}^{(E;b)}$ for the next reclustering step.

- *Ward's hierarchical clustering*

  - It functions as in Framework 1. Here it is invoked on each resampled dissimilarity matrix $\mathbf{D}_{\mathrm{ref}}^{(V;b)}$ for $V \in \{T, E\}$, and outputs the resampled clustering hierarchy for that view and iteration.

- *consensus clustering procedure*

  - It chains the four tools above to resample and recluster the reference sites $B$ times in each view. Across iterations, it accumulates the co-assignment frequency $M_{iq}^{(V)}$ for every pair of sites $(i, q)$, and outputs the co-assignment matrix $\mathbf{M}^{(V)} \in [0,1]^{m_{\mathrm{ref}} \times m_{\mathrm{ref}}}$ for each view $V \in \{T, E\}$.

- *concordance margin calculator*

  - It accepts a co-assignment matrix $\mathbf{M}^{(V)}$ and the anchor partition $\{C_1, C_2, \dots, C_k\}$, then computes the concordance margin $\mu_i^{(V)}$ for each site $i$ as the difference between its mean co-assignment with anchor siblings and the strongest mean co-assignment with non-siblings.

**Interpreters**

- *concordance status assigner*

  - It accepts the per-site margin pair $(\mu_i^{(T)}, \mu_i^{(E)})$, evaluates the sign of each margin, and assigns site $i$ to one of four categories: **concordant**, **taxa-only stable**, **env-only stable**, or **ambiguous**.

#### Outputs

**Results**

- figure: per-site scatter of taxa concordance margin $\mu_i^{(T)}$ against environmental concordance margin $\mu_i^{(E)}$ across the $m_{\mathrm{ref}}$ reference sites
  - x-axis: $\mu_i^{(T)}$, taxa-space margin between sibling and strongest non-sibling co-assignment
  - y-axis: $\mu_i^{(E)}$, environmental-space margin between sibling and strongest non-sibling co-assignment
  - points: each site $i$ at coordinates $(\mu_i^{(T)}, \mu_i^{(E)})$, colored by its anchor cluster label $C_j$
  - annotations: the four sign-pair quadrants are shaded and labeled by concordance category

**Artifacts**

- table <span style="color: blue;">A4</span>: table of only reference sites with their cluster labels and concordance status
  - rows: $(\text{site ID}, \text{cluster label}, \text{taxa margin}, \text{environmental margin}, \text{concordance status})$
  - shape: $m_{\mathrm{ref}} \times 5$
  
#### Demostration Outputs

<div style="width: 60%; max-width: 700px; margin: 0 auto;">

  ![Per-site concordance margin scatter](demos_images/consensus_margin_scatter2.png)

  *Figure: Per-site concordance margin scatter across the $m_{\mathrm{ref}} = 60$ reference sites. Each point is a site plotted at its taxa-space margin $\mu_i^{(T)}$ (x-axis) and environmental-space margin $\mu_i^{(E)}$ (y-axis), colored by its anchor cluster label from Framework 1 (Cluster 1: $n = 35$, Cluster 2: $n = 8$, Cluster 3: $n = 17$). The upper region ($\mu_i^{(E)} \geq 0$, green shading) marks **concordant** sites whose anchor labels are supported under resampling in both views; the lower region ($\mu_i^{(E)} < 0$, orange shading) marks sites with conflicting environmental profiles, where the anchor label holds in taxa space but a rival cluster pulls more strongly in environmental space. Taxa margins concentrate near $\mu_i^{(T)} \approx 0.9$, indicating that anchor labels are largely stable in taxa space, while environmental margins span a much wider range, exposing the cross-view disagreement that the framework is designed to surface.*

</div>

## Framework 3: Training, Diagnosing, and Applying the Environmental Classifier

**Given a site's environmental conditions, which reference community type would be expected in the absence of strong contamination?**

**Questions and Motivations**

The concordant sites identified in Framework 2 form a high-confidence training set: their anchor taxa labels are stable under resampling in both the taxa and environmental views, so the supervised mapping $\mathbf{E} \rightarrow C_j$ can be learned from cases where the two data views agree. Training a classifier on this concordant subset rather than on all reference sites is expected to yield decision boundaries that are anchored on cross-view-supported cases, rather than on sites where the environment-to-taxa relationship is internally inconsistent.

The four concordance categories from Framework 2 play distinct roles in classifier training and diagnostics. Only **concordant** sites — where the anchor label is supported in both views — enter the training set, anchoring the decision boundaries on cross-view-supported cases. The three discordant categories are held out and used as a structured diagnostic set: each category isolates a different failure mode of the environment-to-taxa mapping, so the classifier's performance pattern across the three groups localizes *why* the mapping breaks down rather than merely *how much*.


<div style="width: 80%; max-width: 700px; margin: 0 auto;">

*Table: Role of each concordance category in classifier training and diagnostics.*

| Category | $(\mu^{(T)}, \mu^{(E)})$ | Role | Diagnostic Signal |
|---|---|---|---|
| **Concordant** | $(+, +)$ | Training set | Baseline mapping performance |
| **Taxa-only stable** | $(+, -)$ | Diagnostic | Descriptor insufficiency |
| **Env-only stable** | $(-, +)$ | Diagnostic | Taxa-label fragility |
| **Ambiguous** | $(-, -)$ | Diagnostic | Community transition |

</div>

Once the classifier is finalized, applying it to **all** sites — regardless of contamination level — partitions the environmental descriptor space into $k$ regions, each corresponding to the reference community type expected under low-contamination conditions. For sites in the impacted majority, this provides a counterfactual community type: the community that the environment *would* support if contamination were not driving composition away from the reference state.

**Methodology**

Two candidate classifiers are trained and compared:

- **Linear Discriminant Analysis (LDA)** — assumes Gaussian class-conditional densities with shared covariance, produces linear decision boundaries, and outputs posterior probabilities $P(C_j \mid \mathbf{e}_i)$ for each class.
- **Multivariate Regression Tree (MRT)** — partitions environmental space recursively via axis-aligned splits, producing piecewise-constant decision regions without distributional assumptions.

Both classifiers are trained on the concordant subset $R_{\mathrm{con}}$ from Framework 2, using z-score standardized environmental descriptors as predictors and the anchor cluster labels $C_j$ as response classes. Performance is evaluated through three procedures:

1. **Cross-validation on the concordant training set** — repeated random-split cross-validation produces a distribution of accuracy estimates and an aggregated confusion matrix, characterizing the classifier's internal generalization on cross-view-supported cases.

2. **Diagnostic prediction on discordant sites** — the concordant-trained classifier predicts anchor labels for each of the three discordant categories separately. Per-category confusion matrices and accuracies localize where the environment-to-taxa mapping holds (or fails) outside the concordant core.

3. **Stratified confusion matrix** — predictions are cross-classified by anchor cluster $C_j$ (taxa view) and predicted cluster $\hat{C}_j$ (environmental view), with rows/columns further stratified by taxa-margin sign and environmental-margin sign. This exposes whether classifier errors concentrate in particular margin-sign combinations.

The finalized classifier is then applied to **all** sites in the dataset (not just the reference subset), assigning each site a posterior probability vector $(P(C_1 \mid \mathbf{e}_i), \dots, P(C_k \mid \mathbf{e}_i))$ over the $k$ reference community types.

Rather than forcing each site into a single hard label, the framework retains every class with appreciable posterior support. Site $i$ is assigned to the **set of plausible community types**:

$$
\hat{\mathcal{S}}_i = \left\{ C_j \,:\, P(C_j \mid \mathbf{e}_i) \geq \tau \right\}, \quad \tau = 1/k
$$

The threshold $\tau = 1/k$ is the uniform-prior baseline — any class clearing this threshold carries evidential support beyond no information. For $k = 3$, $\tau \approx 0.333$, so a site enters every cluster whose posterior exceeds 33.3%, and may therefore belong to one, two, or all three clusters simultaneously. A site appearing in $\hat{\mathcal{S}}_i$ for multiple clusters is treated as a genuine member of each — acknowledging that some sites sit in ecological transition zones where more than one reference community type is environmentally plausible.

To visualize the classifier's partition of environmental space, PCA is fit on the standardized environmental descriptors. For each pair of PC axes $(PC_a, PC_b)$, decision regions are projected by holding the remaining PCs at their sample means and predicting $\hat{\mathcal{S}}$ across a dense grid in the $(PC_a, PC_b)$ plane. Regions where $|\hat{\mathcal{S}}| = 1$ are shaded by the single plausible class; regions where $|\hat{\mathcal{S}}| \geq 2$ are rendered with blended shading to mark overlapping community zones. Concordant training sites are overlaid colored by anchor cluster label.

**Conceptual Steps**

1. Extract the concordant subset $R_{\mathrm{con}}$ from the output of Framework 2 and z-score standardize the environmental descriptors using means and standard deviations computed on $\mathbf{E}_{R_{\mathrm{con}}}$. Apply the same transformation parameters to the full environmental matrix $\mathbf{E}$ for downstream prediction.

2. Fit both **LDA** and **MRT** on $(\mathbf{E}_{R_{\mathrm{con}}}', \mathbf{C}_{R_{\mathrm{con}}})$, where $\mathbf{C}_{R_{\mathrm{con}}}$ is the anchor cluster label vector.

3. Run repeated random-split cross-validation on $R_{\mathrm{con}}$ for both classifiers to obtain per-class accuracy distributions and aggregated confusion matrices.

4. Predict anchor labels on each discordant category separately (taxa-only stable, env-only stable, ambiguous) using both classifiers, and construct per-category confusion matrices against the anchor labels.

5. Build the **stratified confusion matrix** cross-classifying predictions by $(\text{taxa margin sign}, \text{env margin sign})$ to locate where classifier errors concentrate in the four-quadrant concordance space.

6. Compare LDA and MRT across the three diagnostic procedures and select the finalized classifier.

7. Apply the finalized classifier to the full standardized environmental matrix $\mathbf{E}'$ to assign each of the 310 sites:
   - a posterior probability vector $(P(C_1 \mid \mathbf{e}_i), \dots, P(C_k \mid \mathbf{e}_i))$;
   - a plausible-class set $\hat{\mathcal{S}}_i = \{C_j : P(C_j \mid \mathbf{e}_i) \geq 1/k\}$ — the site is treated as a member of every cluster in this set.

8. Fit PCA on $\mathbf{E}'$ to obtain ordination axes $PC_1, PC_2, PC_3, \dots$. For each pair $(PC_a, PC_b) \in \{(1,2), (1,3), (2,3)\}$, generate a dense prediction grid by holding remaining PCs at their sample means, back-transform grid points to the original environmental coordinate system, and predict $\hat{C}$ at every grid cell. Color the grid by the predicted class to produce the decision-region map. Overlay concordant sites colored by anchor cluster label.

**Expected Conclusions**

- The classifier trained on concordant sites is expected to achieve high cross-validation accuracy on the concordant subset itself, and the LDA-vs-MRT comparison localizes which model handles the descriptor-to-cluster mapping more effectively for this dataset.

- Diagnostic predictions on the three discordant categories produce a structured performance profile: high accuracy on **taxa-only stable** sites would indicate that environmental descriptors generalize beyond the concordant core; low accuracy there would confirm the descriptor-insufficiency signal flagged in Framework 2.

- The PCA-projected decision regions reveal how the finalized classifier partitions the dominant environmental gradients into $k$ community-type regions, and overlaying the concordant training sites shows the geometric basis on which these regions were learned.

### Computation Process

#### Inputs

**Metadata**

- Environmental data $\mathbf{E}$
- Taxa data $\mathbf{T}$

**Artifacts**

- table <span style="color: blue;">A4</span>: reference subset of the augmented table with contamination scores, reference subset membership, reference cluster labels, and concordance status
  - rows: $(\text{site ID}, \text{cluster label},  \text{taxa margin}, \text{environmental margin}, \text{concordance status})$
  - shape: $m_{\mathrm{ref}} \times 5$
  - source: output artifact from the consensus clustering framework

#### Process

**Transformations**

- *z-score standardizer*

  - It accepts the environmental descriptor matrix $\mathbf{E}$ and applies z-score standardization with means and standard deviations computed on the concordant subset $R_{\mathrm{con}}$, producing $\mathbf{E}'$ for both training and full-dataset prediction.

- *PCA ordinator*

  - It accepts the standardized environmental matrix $\mathbf{E}'$ and fits a principal components decomposition (same as PCA on the correlation matrix of $\mathbf{E}$), outputting site scores on $PC_1, PC_2, PC_3, \dots$ for visualization of decision regions in two-dimensional ordination space.

**Bridging tools**

- *boolean subset selector*

  - It accepts a dataframe with at least a boolean column $\text{concordance status}$, selects rows by the specified status value, and outputs the corresponding subset $R_{\mathrm{con}}$ or each discordant category.

- *prediction grid generator*

  - It accepts two chosen PC axes $(PC_a, PC_b)$, holds the remaining PCs at their sample means, and outputs a dense grid of points in the $(PC_a, PC_b)$ plane back-transformed into the original environmental coordinate system, ready for classifier prediction.

**Modeling**

- *Linear Discriminant Analysis (LDA) trainer*

  - It accepts $(\mathbf{E}_{R_{\mathrm{con}}}', \mathbf{C}_{R_{\mathrm{con}}})$ and fits a linear discriminant model, outputting the fitted LDA classifier that maps environmental descriptors to posterior probabilities over the $k$ cluster classes.

- *Multivariate Regression Tree (MRT) trainer*

  - It accepts $(\mathbf{E}_{R_{\mathrm{con}}}', \mathbf{C}_{R_{\mathrm{con}}})$ and fits a recursive partitioning tree, outputting the fitted MRT classifier that maps environmental descriptors to predicted cluster classes via axis-aligned splits.

- *random-split cross-validator*

  - It accepts a fitted classifier and a training subset, then performs repeated random splits into train/test partitions, refitting the classifier on each split and aggregating predictions into a cross-validation confusion matrix and per-class accuracy distribution.

- *classifier predictor*

  - It accepts a fitted classifier and an arbitrary site subset, returning predicted cluster labels $\hat{C}_i$ and posterior probability vectors $(P(C_1 \mid \mathbf{e}_i), \dots, P(C_k \mid \mathbf{e}_i))$.

**Interpreters**

- *concordance-stratified confusion matrix*

  - It accepts the per-site true and predicted labels along with each site's margin signs $(\mathrm{sgn}(\mu_i^{(T)}), \mathrm{sgn}(\mu_i^{(E)}))$, and constructs a confusion matrix stratified by the four taxa–environment margin combinations.

- *threshold-based multi-membership assigner*

  - It accepts the posterior probability matrix from the classifier predictor and a threshold $\tau$ (default $1/k$), then returns the plausible-class set $\hat{\mathcal{S}}_i$ for each site. Sites with $|\hat{\mathcal{S}}_i| \geq 2$ are assigned to multiple clusters simultaneously.

- *PCA decision region projector*

  - It accepts the finalized classifier, the PCA ordination of $\mathbf{E}'$, and a chosen pair of PC axes, then predicts the classifier output across the prediction grid and renders a colored decision-region map of expected community types in the chosen 2D PC plane.


#### Outputs

Assuming $k=3$, the outputs will be initialized as follows:

**Results**

- table: cross-validation confusion matrix on the concordant subset for LDA and MRT
  - rows: $(\text{classifier}, \hat{C}_1, \hat{C}_2, \hat{C}_3) \mid C_j, j \in {1,2,3}$
  - shape: $(2 \times k) \times k$

- table: concordance-stratified confusion matrix for the finalized classifier
  - rows: $(\text{env-margin}, (\hat{C}_1, \hat{C}_2, \hat{C}_3)| \text{taxa-margin} \geq 0, (\hat{C}_1, \hat{C}_2, \hat{C}_3)| \text{taxa-margin} < 0)$
  - shape: $(2 \times k) \times (2 \times k)$

- figure: classifier decision regions in $(PC_1, PC_2)$ ordination space, conditional on means of remaining PCs
  - x-axis: $PC_1$
  - y-axis: $PC_2$
  - background: grid cells colored by the finalized classifier's predicted community type $\hat{C}$
  - points: concordant training sites colored by anchor cluster label $C_j$; discordant sites overlaid with distinct markers indicating concordance status
  - legend: colors for the $k$ community-type regions; marker shapes for concordance status

- figure: classifier decision regions in $(PC_1, PC_3)$ ordination space, conditional on means of remaining PCs
  - x-axis: $PC_1$
  - y-axis: $PC_3$
  - background, points, legend: as above

- figure: classifier decision regions in $(PC_2, PC_3)$ ordination space, conditional on means of remaining PCs
  - x-axis: $PC_2$
  - y-axis: $PC_3$
  - background, points, legend: as above

**Artifacts**

- table <span style="color: blue;">A5</span>: full-site table with posterior probabilities and plausible-class memberships
  - rows: $(\text{site ID}, P(C_1 \mid \mathbf{e}_i), P(C_2 \mid \mathbf{e}_i), P(C_3 \mid \mathbf{e}_i), \hat{\mathcal{S}}_i)$
  - shape: $310 \times 5$

#### Demonstration Outputs

<div align="center">

<div style="width:85%; text-align:left; font-size:13px; margin-bottom:8px;">
  <b>Table: </b> Stratified confusion matrix of classifier predictions under taxa-margin and environmental-margin status combinations. Rows are grouped by environmental margin status, columns are grouped by taxa margin status, and accuracy is reported separately within each taxa–environment margin category.
</div>

<table style="width:85%; border-collapse:collapse; text-align:center; font-size:13px;">
<thead>
<tr style="border-top:3px solid black;">
<td rowspan="2" style="padding:10px; border-bottom:3px solid black;"><b>Env-/Taxa- <br>Margin Status</b></td>
<td rowspan="2" style="padding:10px; border-bottom:3px solid black;"><b>True/Pred</b></td>
<td colspan="3" style="padding:10px; border-bottom:1px solid black;"><b>Non-negative Taxa Margin</b></td>
<td colspan="3" style="padding:10px; border-bottom:1px solid black;"><b>Negative Taxa Margin</b></td>
</tr>
<tr style="border-bottom:3px solid black;">
<td style="padding:10px;">Cluster C<sub>1</sub></td>
<td style="padding:10px;">Cluster C<sub>2</sub></td>
<td style="padding:10px;">Cluster C<sub>3</sub></td>
<td style="padding:10px;">Cluster C<sub>1</sub></td>
<td style="padding:10px;">Cluster C<sub>2</sub></td>
<td style="padding:10px;">Cluster C<sub>3</sub></td>
</tr>
</thead>
<tbody>
<tr>
<td rowspan="4" style="padding:10px; vertical-align:middle;"><b>Non-negative<br>Env Margin</b></td>
<td style="padding:8px;">Cluster C<sub>1</sub></td>
<td>5</td><td>0</td><td>0</td>
<td>0</td><td>0</td><td>0</td>
</tr>
<tr>
<td style="padding:8px;">Cluster C<sub>2</sub></td>
<td>0</td><td>12</td><td>2</td>
<td>0</td><td>0</td><td>0</td>
</tr>
<tr>
<td style="padding:8px;">Cluster C<sub>3</sub></td>
<td>0</td><td>1</td><td>14</td>
<td>0</td><td>0</td><td>1</td>
</tr>
<tr style="border-bottom:3px solid black;">
<td style="padding:8px;"><b>Accuracy</b></td>
<td colspan="3"><b>91% (n=34)</b></td>
<td colspan="3"><b>100% (n=1)</b></td>
</tr>
<tr>
<td rowspan="4" style="padding:10px; vertical-align:middle;"><b>Negative<br>Env Margin</b></td>
<td style="padding:8px;">Cluster C<sub>1</sub></td>
<td>0</td><td>3</td><td>1</td>
<td>0</td><td>0</td><td>0</td>
</tr>
<tr>
<td style="padding:8px;">Cluster C<sub>2</sub></td>
<td>1</td><td>2</td><td>19</td>
<td>0</td><td>0</td><td>0</td>
</tr>
<tr>
<td style="padding:8px;">Cluster C<sub>3</sub></td>
<td>1</td><td>0</td><td>0</td>
<td>0</td><td>0</td><td>0</td>
</tr>
<tr style="border-bottom:3px solid black;">
<td style="padding:8px;"><b>Accuracy</b></td>
<td colspan="3"><b>7% (n=27)</b></td>
<td colspan="3"><b>n=0</b></td>
</tr>
</tbody>
</table>
</div>


<div style="width: 80%; max-width: 700px; margin: 0 auto;">

  ![Decision regions in PCA-projected environmental space](demos_images/ch3_pc123_ordination_projection.png)

  *Figure: Classifier decision regions projected onto the dominant environmental gradients. The finalized classifier is trained on concordant reference sites and applied across a dense grid in each pair of principal component axes $(PC_1, PC_2)$, $(PC_1, PC_3)$, $(PC_2, PC_3)$ — with remaining PCs held at their sample means. Background shading colors each grid cell by the predicted expected community type $\hat{C}$, partitioning environmental space into $k = 3$ regions. Concordant training sites are overlaid as filled points colored by their anchor cluster label $C_j$, showing the cross-view-supported cases that anchored the boundaries; discordant sites are overlaid with distinct markers indicating their concordance status, exposing where the classifier's environmental partition disagrees with the taxa-defined labels.*

</div>